In [168]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score
from sklearn.model_selection import cross_val_score

In [169]:
df = pd.read_csv("housing.csv")

In [170]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [171]:
x = df.drop('median_house_value', axis=1) 
y = df['median_house_value']

In [172]:
x_train, x_test, y_train, y_test = train_test_split(x,y, random_state=42, test_size=0.2)

In [173]:
num_att = x.drop('ocean_proximity', axis=1).columns.to_list()
cat_att = ['ocean_proximity']

In [174]:
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')), 
    ('stand', StandardScaler())
])
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder())
])


final = ColumnTransformer([
    ('num', num_pipe, num_att), 
    ('cat', cat_pipe, cat_att)
])

In [175]:
x_trans = final.fit_transform(x_train)

In [176]:
model = RandomForestRegressor()
model = model.fit(x_trans, y_train)

In [177]:
test_trans = final.transform(x_test)

In [178]:
model.score(test_trans, y_test)

0.8156905542399073

In [179]:
# Hyperparameter tuning is the process of selecting optimal model parameters using 
# techniques like GridSearchCV or RandomizedSearchCV combined with cross-validation to 
# improve model performance.

# number of trees in random forest 
n_estimators = [20, 60, 100, 120]

# number of features to consider at every split
max_features = [0.2, 0.6, 1.0]

# maximum number of levels in tree
max_depth = [2, 8, None]

# number of samples
max_samples = [0.5, 0.75, 1.0]



# in this as we are using 4 estimators and 3 features etc. (4*3*3*3 = 108) so there will 
# be 108 different random forest for training (4)

In [180]:
rcv = -cross_val_score(model, test_trans, y_test, cv=10, scoring='neg_root_mean_squared_error')
rcv = pd.Series(rcv).describe()

In [181]:
rcv

count       10.000000
mean     56415.840056
std       4821.066491
min      47379.251417
25%      53420.163265
50%      57541.155796
75%      60599.914691
max      61521.378288
dtype: float64

In [182]:
np.mean(cross_val_score(model, test_trans, y_test, cv=10, scoring='r2'))
# as wee can see our model acuuracy was 81% but after applying the cross validation
# it be 75.5% so we can tune our model for better prediction. 

np.float64(0.7554061770836715)

# GridsearchCV

In [145]:
param_grid = {'n_estimators' : n_estimators, 
              'max_features' : max_features, 
              'max_depth' : max_depth, 
              'max_samples' : max_samples
}

In [183]:
rf = RandomForestRegressor()

In [147]:
from sklearn.model_selection import GridSearchCV
rf_grid = GridSearchCV(estimator=rf,  # choose model 
                      param_grid=param_grid, 
                      cv=5, 
                      verbose=2, #it shows output during the process
                      n_jobs=-1 # it uses all cores which help to fast the process(how many CPU cores to use), 
                      )

In [150]:
new_model = rf_grid.fit(x_trans, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


In [153]:
new_model.best_params_

{'max_depth': None,
 'max_features': 0.6,
 'max_samples': 1.0,
 'n_estimators': 100}

In [155]:
new_model.best_score_

np.float64(0.8212869963795892)

# RandomSearchCV

In [189]:
# GridSearch is Very slow but good for small dataset if we need fast result then we uses 
# RandomSearchCv it will take Random combinations and faster than gridsearch.


# number of trees in random forest 
n_estimators = [20, 60, 100, 120]

# number of features to consider at every split
max_features = [0.2, 0.6, 1.0]

# maximum number of levels in tree
max_depth = [2, 8, None]

# number of samples
max_samples = [0.5, 0.75, 1.0]

# bootstrap sample 
bootstrap = [True] 

# min no. of sample required to split the node 
min_samples_split = [2,5]

# min no. of sample required as each leaf node 
min_samples_leaf = [1,2] 


In [190]:
param_grid = {'n_estimators' : n_estimators, 
              'max_features' : max_features, 
              'max_depth' : max_depth, 
              'max_samples' : max_samples,
              'bootstrap' : bootstrap, 
              'min_samples_split' : min_samples_split, 
              'min_samples_leaf' : min_samples_leaf
}

In [191]:
from sklearn.model_selection import RandomizedSearchCV

In [192]:
rs = RandomizedSearchCV(estimator=rf, 
                       param_distributions= param_grid, 
                       cv=5, 
                       verbose=2, 
                       n_jobs=-1)

In [193]:
nmode = rs.fit(x_trans, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
